In [1]:
import os
import json
import shutil
from tqdm import tqdm
from pycocotools.coco import COCO

In [2]:
# Paths (relative to this notebook)
COCO_ANN = "../datasets/yolo_exam/coco_raw/annotations/instances_train2017.json"
COCO_IMG_DIR = "../datasets/yolo_exam/coco_raw/train2017"

OUT_IMG_DIR = "../datasets/yolo_exam/images/train"
OUT_LBL_DIR = "../datasets/yolo_exam/labels/train"

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LBL_DIR, exist_ok=True)

# COCO → YOLO class mapping
TARGET_CLASSES = {
    1: 0,   # person
    67: 1   # cell phone
}

In [3]:
coco = COCO(COCO_ANN)
img_ids = coco.getImgIds(catIds=list(TARGET_CLASSES.keys()))

print("Total images:", len(img_ids))

for img_id in tqdm(img_ids):
    img = coco.loadImgs(img_id)[0]
    img_path = os.path.join(COCO_IMG_DIR, img["file_name"])

    if not os.path.exists(img_path):
        continue

    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
    if not anns:
        continue

    shutil.copy(img_path, os.path.join(OUT_IMG_DIR, img["file_name"]))

    label_lines = []

    for ann in anns:
        cat_id = ann["category_id"]
        if cat_id not in TARGET_CLASSES:
            continue

        x, y, w, h = ann["bbox"]
        img_w, img_h = img["width"], img["height"]

        xc = (x + w / 2) / img_w
        yc = (y + h / 2) / img_h
        wn = w / img_w
        hn = h / img_h

        label_lines.append(
            f"{TARGET_CLASSES[cat_id]} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}"
        )

    if label_lines:
        label_path = os.path.join(
            OUT_LBL_DIR, img["file_name"].replace(".jpg", ".txt")
        )
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

loading annotations into memory...
Done (t=16.63s)
creating index...
index created!
Total images: 5869


100%|██████████| 5869/5869 [00:29<00:00, 200.48it/s]
